Step 2 – Upload the processed datasets

In [1]:
from google.colab import files

uploaded = files.upload()

Saving portuguese_test_processed.csv to portuguese_test_processed.csv
Saving portuguese_train_processed.csv to portuguese_train_processed.csv


Step 3 – Import required libraries

In [ ]:
import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from IPython.display import display

from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor

from sklearn.model_selection import (
    KFold,
    cross_validate,
    cross_val_predict
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("All libraries imported successfully!")

Step 4 – Create output folders

In [3]:
from pathlib import Path

project_dir = Path("/content/Portuguese_ML_Project")

models_dir = project_dir / "models"

results_dir = (
    project_dir / "output" /
    "ridge_regression" / "comparison"
)

models_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

print("Project folders created successfully!")

Project folders created successfully!


Step 5 – Load training and testing datasets

In [5]:
train_df = pd.read_csv(
    "/content/portuguese_train_processed.csv"
)

test_df = pd.read_csv(
    "/content/portuguese_test_processed.csv"
)

X_train = train_df.drop(columns=["G3"])
y_train = train_df["G3"]

X_test = test_df.drop(columns=["G3"])
y_test = test_df["G3"]

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

assert X_train.columns.equals(X_test.columns)
assert X_train.notna().all().all()
assert y_train.notna().all()
assert X_test.notna().all().all()
assert y_test.notna().all()

display(train_df.head())

Training Shape: (519, 58)
Testing Shape: (130, 58)
Training Target: (519,)
Testing Target: (130,)


,num__age,num__Medu,num__Fedu,num__traveltime,num__studytime,num__failures,num__famrel,num__freetime,num__goout,num__Dalc,...,cat__activities_yes,cat__nursery_no,cat__nursery_yes,cat__higher_no,cat__higher_yes,cat__internet_no,cat__internet_yes,cat__romantic_no,cat__romantic_yes,G3
0,0.987932,-0.437418,-0.256337,-0.754310,1.352962,-0.382133,0.068136,-0.186680,-0.196270,-0.539674,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,18
1,-0.629534,1.350140,1.591423,-0.754310,0.121054,-0.382133,0.068136,0.753970,1.501468,3.579309,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,12
2,0.987932,0.456361,-0.256337,-0.754310,1.352962,-0.382133,1.108214,-0.186680,-1.045139,-0.539674,...,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,12
3,0.179199,-0.437418,-1.180216,-0.754310,-1.110853,-0.382133,0.068136,0.753970,-1.045139,0.490072,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,13
4,0.179199,-1.331196,-1.180216,0.555011,-1.110853,-0.382133,-0.971942,1.694619,1.501468,0.490072,...,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,10


Step 6 – Define five Ridge Regression experiments

In [6]:
ridge_experiments = {
    "Ridge-1": 0.01,
    "Ridge-2": 0.1,
    "Ridge-3": 1.0,
    "Ridge-4": 10.0,
    "Ridge-5": 100.0
}

experiment_df = pd.DataFrame({
    "Experiment": list(ridge_experiments.keys()),
    "Alpha": list(ridge_experiments.values())
})

display(experiment_df)

,Experiment,Alpha
0,Ridge-1,0.01
1,Ridge-2,0.10
2,Ridge-3,1.00
3,Ridge-4,10.00
4,Ridge-5,100.00


Step 7 – Configure five-fold cross-validation

In [8]:
from sklearn.model_selection import KFold

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "R2": "r2"
}

print("5-Fold Cross-Validation configured!")

5-Fold Cross-Validation configured!


Step 8 – Train all Ridge configurations

In [12]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

ridge_results = []
ridge_fold_results = {}
ridge_predictions = {}

actual = y_train.to_numpy()
actual_binary = (actual >= 10).astype(int)

for name, alpha in ridge_experiments.items():

    print(f"Training {name} | Alpha = {alpha}")

    model = Ridge(alpha=alpha)

    # Cross-validation
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        error_score="raise"
    )

    # Out-of-fold predictions
    predictions = cross_val_predict(
        Ridge(alpha=alpha),
        X_train,
        y_train,
        cv=cv
    )

    ridge_predictions[name] = predictions

    # Round predicted grades
    rounded_predictions = np.clip(
        np.rint(predictions),
        0,
        20
    ).astype(int)

    # Convert grades to Pass / Fail
    predicted_binary = (
        predictions >= 10
    ).astype(int)

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(
        actual_binary,
        predicted_binary,
        labels=[0, 1]
    ).ravel()

    # Regression errors
    train_rmse = np.sqrt(
        -scores["train_MSE"]
    ).mean()

    val_rmse = np.sqrt(
        -scores["test_MSE"]
    ).mean()

    ridge_results.append({
        "Experiment": name,
        "Alpha": alpha,

        # Regression metrics
        "Train MAE": -scores["train_MAE"].mean(),
        "Validation MAE": -scores["test_MAE"].mean(),

        "Train MSE": -scores["train_MSE"].mean(),
        "Validation MSE": -scores["test_MSE"].mean(),

        "Train RMSE": train_rmse,
        "Validation RMSE": val_rmse,

        "Train R2": scores["train_R2"].mean(),
        "Validation R2": scores["test_R2"].mean(),

        "RMSE Gap": val_rmse - train_rmse,

        # Grade prediction accuracy
        "Exact Grade Accuracy (%)": (
            np.mean(
                rounded_predictions == actual
            ) * 100
        ),

        "Within 1 Grade (%)": (
            np.mean(
                np.abs(actual - predictions) <= 1
            ) * 100
        ),

        "Within 2 Grades (%)": (
            np.mean(
                np.abs(actual - predictions) <= 2
            ) * 100
        ),

        # Classification metrics
        "Pass/Fail Accuracy (%)": (
            accuracy_score(
                actual_binary,
                predicted_binary
            ) * 100
        ),

        "Precision": precision_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        "Recall": recall_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        "Specificity": (
            tn / (tn + fp)
            if (tn + fp) > 0 else 0
        ),

        "F1 Score": f1_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        # Confusion matrix
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

    ridge_fold_results[name] = scores

ridge_results_df = pd.DataFrame(ridge_results)

ridge_results_df = ridge_results_df.sort_values(
    by=[
        "Validation RMSE",
        "Validation MAE",
        "RMSE Gap"
    ],
    ascending=[True, True, True]
).reset_index(drop=True)

print("\nAll Ridge experiments completed!")

display(ridge_results_df.round(4))

Training Ridge-1 | Alpha = 0.01
Training Ridge-2 | Alpha = 0.1
Training Ridge-3 | Alpha = 1.0
Training Ridge-4 | Alpha = 10.0
Training Ridge-5 | Alpha = 100.0

All Ridge experiments completed!


,Experiment,Alpha,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,Validation R2,...,Within 2 Grades (%),Pass/Fail Accuracy (%),Precision,Recall,Specificity,F1 Score,TN,FP,FN,TP
0,Ridge-4,10.00,0.7868,0.8698,1.4651,1.8275,1.2080,1.3237,0.8611,0.8284,...,93.8343,87.6686,0.9793,0.8710,0.9059,0.9220,77,8,56,378
1,Ridge-3,1.00,0.7905,0.8826,1.4545,1.8403,1.2037,1.3306,0.8621,0.8266,...,93.8343,87.6686,0.9818,0.8687,0.9176,0.9218,78,7,57,377
2,Ridge-2,0.10,0.7913,0.8844,1.4544,1.8433,1.2036,1.3320,0.8621,0.8262,...,93.8343,87.6686,0.9818,0.8687,0.9176,0.9218,78,7,57,377
3,Ridge-1,0.01,0.7914,0.8846,1.4544,1.8437,1.2036,1.3322,0.8621,0.8261,...,93.8343,87.6686,0.9818,0.8687,0.9176,0.9218,78,7,57,377
4,Ridge-5,100.00,0.8223,0.8812,1.7259,2.0219,1.3117,1.3861,0.8363,0.8122,...,93.4489,89.9807,0.9823,0.8963,0.9176,0.9373,78,7,45,389


Step 9 – Compare regression errors

In [13]:
regression_columns = [
    "Experiment",
    "Alpha",
    "Train MAE",
    "Validation MAE",
    "Train MSE",
    "Validation MSE",
    "Train RMSE",
    "Validation RMSE",
    "Train R2",
    "Validation R2",
    "RMSE Gap"
]

print("=" * 65)
print("RIDGE REGRESSION – ERROR COMPARISON")
print("=" * 65)

display(
    ridge_results_df[regression_columns].round(4)
)

RIDGE REGRESSION – ERROR COMPARISON


,Experiment,Alpha,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,Validation R2,RMSE Gap
0,Ridge-4,10.00,0.7868,0.8698,1.4651,1.8275,1.2080,1.3237,0.8611,0.8284,0.1157
1,Ridge-3,1.00,0.7905,0.8826,1.4545,1.8403,1.2037,1.3306,0.8621,0.8266,0.1270
2,Ridge-2,0.10,0.7913,0.8844,1.4544,1.8433,1.2036,1.3320,0.8621,0.8262,0.1284
3,Ridge-1,0.01,0.7914,0.8846,1.4544,1.8437,1.2036,1.3322,0.8621,0.8261,0.1286
4,Ridge-5,100.00,0.8223,0.8812,1.7259,2.0219,1.3117,1.3861,0.8363,0.8122,0.0744


Step 10 – Compare grade prediction accuracy

In [14]:
grade_columns = [
    "Experiment",
    "Alpha",
    "Exact Grade Accuracy (%)",
    "Within 1 Grade (%)",
    "Within 2 Grades (%)"
]

print("=" * 65)
print("RIDGE REGRESSION – GRADE ACCURACY")
print("=" * 65)

display(
    ridge_results_df[grade_columns].round(2)
)

RIDGE REGRESSION – GRADE ACCURACY


,Experiment,Alpha,Exact Grade Accuracy (%),Within 1 Grade (%),Within 2 Grades (%)
0,Ridge-4,10.00,38.54,70.52,93.83
1,Ridge-3,1.00,38.73,70.71,93.83
2,Ridge-2,0.10,39.31,70.71,93.83
3,Ridge-1,0.01,39.11,70.71,93.83
4,Ridge-5,100.00,39.69,71.68,93.45


Step 11 – Compare classification metrics

In [15]:
classification_columns = [
    "Experiment",
    "Alpha",
    "Pass/Fail Accuracy (%)",
    "Precision",
    "Recall",
    "Specificity",
    "F1 Score"
]

print("=" * 65)
print("RIDGE REGRESSION – CLASSIFICATION COMPARISON")
print("=" * 65)

display(
    ridge_results_df[classification_columns].round(4)
)

RIDGE REGRESSION – CLASSIFICATION COMPARISON


,Experiment,Alpha,Pass/Fail Accuracy (%),Precision,Recall,Specificity,F1 Score
0,Ridge-4,10.00,87.6686,0.9793,0.8710,0.9059,0.9220
1,Ridge-3,1.00,87.6686,0.9818,0.8687,0.9176,0.9218
2,Ridge-2,0.10,87.6686,0.9818,0.8687,0.9176,0.9218
3,Ridge-1,0.01,87.6686,0.9818,0.8687,0.9176,0.9218
4,Ridge-5,100.00,89.9807,0.9823,0.8963,0.9176,0.9373


Step 12 – Compare confusion matrix values

In [16]:
confusion_columns = [
    "Experiment",
    "Alpha",
    "TN",
    "FP",
    "FN",
    "TP"
]

print("=" * 65)
print("RIDGE REGRESSION – CONFUSION MATRIX COMPARISON")
print("=" * 65)

display(
    ridge_results_df[confusion_columns]
)

RIDGE REGRESSION – CONFUSION MATRIX COMPARISON


,Experiment,Alpha,TN,FP,FN,TP
0,Ridge-4,10.00,77,8,56,378
1,Ridge-3,1.00,78,7,57,377
2,Ridge-2,0.10,78,7,57,377
3,Ridge-1,0.01,78,7,57,377
4,Ridge-5,100.00,78,7,45,389


Step 13 – Check overfitting and underfitting

In [18]:
from sklearn.dummy import DummyRegressor
import numpy as np

baseline = DummyRegressor(strategy="mean")

baseline_scores = cross_validate(
    baseline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

baseline_rmse = np.sqrt(
    -baseline_scores["test_MSE"]
).mean()

def assess_fitting(row):

    gap_ratio = (
        row["RMSE Gap"] / row["Train RMSE"]
        if row["Train RMSE"] > 0
        else np.inf
    )

    if row["Validation RMSE"] >= baseline_rmse:
        return "Weak performance / possible underfitting"

    elif gap_ratio > 0.30:
        return "Possible overfitting"

    else:
        return "Reasonable generalization"

ridge_results_df["Fitting Assessment"] = (
    ridge_results_df.apply(
        assess_fitting,
        axis=1
    )
)

print("Baseline RMSE:", round(baseline_rmse, 4))

display(
    ridge_results_df[
        [
            "Experiment",
            "Alpha",
            "Train RMSE",
            "Validation RMSE",
            "RMSE Gap",
            "Fitting Assessment"
        ]
    ].round(4)
)

Baseline RMSE: 3.2326


,Experiment,Alpha,Train RMSE,Validation RMSE,RMSE Gap,Fitting Assessment
0,Ridge-4,10.00,1.2080,1.3237,0.1157,Reasonable generalization
1,Ridge-3,1.00,1.2037,1.3306,0.1270,Reasonable generalization
2,Ridge-2,0.10,1.2036,1.3320,0.1284,Reasonable generalization
3,Ridge-1,0.01,1.2036,1.3322,0.1286,Reasonable generalization
4,Ridge-5,100.00,1.3117,1.3861,0.0744,Reasonable generalization



Step 14 – Final Ridge Regression comparison

In [19]:
final_comparison = ridge_results_df.copy()

final_comparison.insert(
    0,
    "Rank",
    np.arange(1, len(final_comparison) + 1)
)

print("=" * 70)
print("FINAL PORTUGUESE RIDGE REGRESSION COMPARISON")
print("=" * 70)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 250
):
    display(final_comparison.round(4))

FINAL PORTUGUESE RIDGE REGRESSION COMPARISON


,Rank,Experiment,Alpha,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,Validation R2,RMSE Gap,Exact Grade Accuracy (%),Within 1 Grade (%),Within 2 Grades (%),Pass/Fail Accuracy (%),Precision,Recall,Specificity,F1 Score,TN,FP,FN,TP,Fitting Assessment
0,1,Ridge-4,10.00,0.7868,0.8698,1.4651,1.8275,1.2080,1.3237,0.8611,0.8284,0.1157,38.5356,70.5202,93.8343,87.6686,0.9793,0.8710,0.9059,0.9220,77,8,56,378,Reasonable generalization
1,2,Ridge-3,1.00,0.7905,0.8826,1.4545,1.8403,1.2037,1.3306,0.8621,0.8266,0.1270,38.7283,70.7129,93.8343,87.6686,0.9818,0.8687,0.9176,0.9218,78,7,57,377,Reasonable generalization
2,3,Ridge-2,0.10,0.7913,0.8844,1.4544,1.8433,1.2036,1.3320,0.8621,0.8262,0.1284,39.3064,70.7129,93.8343,87.6686,0.9818,0.8687,0.9176,0.9218,78,7,57,377,Reasonable generalization
3,4,Ridge-1,0.01,0.7914,0.8846,1.4544,1.8437,1.2036,1.3322,0.8621,0.8261,0.1286,39.1137,70.7129,93.8343,87.6686,0.9818,0.8687,0.9176,0.9218,78,7,57,377,Reasonable generalization
4,5,Ridge-5,100.00,0.8223,0.8812,1.7259,2.0219,1.3117,1.3861,0.8363,0.8122,0.0744,39.6917,71.6763,93.4489,89.9807,0.9823,0.8963,0.9176,0.9373,78,7,45,389,Reasonable generalization


Step 15 – Select the best Ridge configuration

In [20]:
best_result = final_comparison.iloc[0]

best_name = best_result["Experiment"]
best_alpha = float(best_result["Alpha"])

print("=" * 60)
print("BEST PORTUGUESE RIDGE REGRESSION CONFIGURATION")
print("=" * 60)

print("Experiment:", best_name)
print("Alpha:", best_alpha)
print("Number of Features:", X_train.shape[1])

print("\nREGRESSION METRICS")
print(f"Validation RMSE : {best_result['Validation RMSE']:.4f}")
print(f"Validation MAE  : {best_result['Validation MAE']:.4f}")
print(f"Validation R²   : {best_result['Validation R2']:.4f}")
print(f"RMSE Gap        : {best_result['RMSE Gap']:.4f}")

print("\nGRADE ACCURACY")
print(f"Exact Accuracy  : {best_result['Exact Grade Accuracy (%)']:.2f}%")
print(f"Within ±1 Grade : {best_result['Within 1 Grade (%)']:.2f}%")
print(f"Within ±2 Grades: {best_result['Within 2 Grades (%)']:.2f}%")

print("\nPASS/FAIL CLASSIFICATION")
print(f"Accuracy   : {best_result['Pass/Fail Accuracy (%)']:.2f}%")
print(f"Precision  : {best_result['Precision']:.4f}")
print(f"Recall     : {best_result['Recall']:.4f}")
print(f"Specificity: {best_result['Specificity']:.4f}")
print(f"F1 Score   : {best_result['F1 Score']:.4f}")

print("\nCONFUSION MATRIX")
print("TN:", int(best_result["TN"]))
print("FP:", int(best_result["FP"]))
print("FN:", int(best_result["FN"]))
print("TP:", int(best_result["TP"]))

print("\nFITTING ASSESSMENT")
print(best_result["Fitting Assessment"])

BEST PORTUGUESE RIDGE REGRESSION CONFIGURATION
Experiment: Ridge-4
Alpha: 10.0
Number of Features: 58

REGRESSION METRICS
Validation RMSE : 1.3237
Validation MAE  : 0.8698
Validation R²   : 0.8284
RMSE Gap        : 0.1157

GRADE ACCURACY
Exact Accuracy  : 38.54%
Within ±1 Grade : 70.52%
Within ±2 Grades: 93.83%

PASS/FAIL CLASSIFICATION
Accuracy   : 87.67%
Precision  : 0.9793
Recall     : 0.8710
Specificity: 0.9059
F1 Score   : 0.9220

CONFUSION MATRIX
TN: 77
FP: 8
FN: 56
TP: 378

FITTING ASSESSMENT
Reasonable generalization
